In [1]:
import torch
import numpy as np
import pandas as pd
from utils_sample import *

In [2]:
policy=torch.load('./knn_rnn.pth')

In [3]:
h_state = np.load(f"h_state.npy")
h_state = torch.Tensor(h_state)
SRL = torch.load('./rnn.pth')

In [4]:
#select 3 patient example N110859,N393007,N467846

In [5]:
age_gen=pd.DataFrame([[ 'N110859', 5, 2],
       ['N393007', 4, 1],
       ['N467846', 4, 2]],columns=["patient_id", "age","gender"])
his_t=pd.DataFrame([['N110859', 1, 1, 0, 1, 1],
       ['N393007', 1, 0, 0, 0, 1],
       ['N467846', 1, 1, 0, 0, 1]],columns=['id','ckd', 'cag', '慢性肺部疾病', 'md', 'ht'])

In [ ]:
def action_pred(SRL,h_state,act,s_l,policy):
    hid_s=info_rep(SRL,h_state,'rnn',act[:-1,:],s_l[:-1,:])
    p=torch.zeros([1,16])
    
    hid_s=torch.concat([p, hid_s],dim=0)
    s_l = torch.where(
                    torch.isnan(s_l),
                    torch.full_like(s_l, 0), s_l)
    state = torch.concat([s_l, hid_s], dim=-1)
    cla = ''
    for j in range(1,7):
        cla += str(int(s_l[0][j].numpy()))
    idx_close = policy.info_cla[cla]
    action_set=torch.unique(torch.index_select(policy.train_all_action, 0, torch.tensor(idx_close)),dim=0)
    action_set_trans = torch.where(action_set == 0, -1, action_set)
    action_re=policy.get_action_eval(state)
    
    action_trans = torch.where(action_re == 0, -1, action_re.long())

    return state,action_re,action_trans

In [24]:
for patient_id in ["N110859","N393007","N467846"]:
    state = pd.read_csv(test_path + patient_id + '/' + 'state.csv')
    action = pd.read_csv(test_path + patient_id + '/' + 'action.csv')
    test = pd.read_csv(test_path + patient_id  + '/' + 'test.csv')
    test['ind'] = np.arange(len(test))
    test_dic = dict(test[['Unnamed: 0', 'ind']].values)
    
    s_l=[]
    a_l=[]
    for i in range(len(state)):
        s,scr=gen_state(state.iloc[i,:])

        date = state.loc[i, 'date']
        ind_=test_dic[date]
        test_s=test.iloc[ind_,:]
        s=add_s(test_s,s)
        age=age_gen[age_gen['patient_id']==patient_id]['age'].tolist()[0]
        gen=age_gen[age_gen['patient_id']==patient_id]['gender'].tolist()[0]
        ckd=ckd_cal(scr,age,gen)

        s.append(ckd)
        s_his=[]
        s_his.append(age)
        s_his+=his_t[his_t['id']==patient_id][['ckd', 'cag', '慢性肺部疾病', 'md', 'ht']].values[0].tolist()
        s_his.append(gen)
        s=s_his+s
        s_l.append(s)

        a=action.iloc[i,1:].tolist()

        a+=np.zeros(5).tolist()
        a_l.append(a)
    s_l=torch.tensor(s_l).float()
    act=torch.tensor(a_l)
    #RL_HFRx_act is RL_HFRx's medication
    state,RL_HFRx_act,RL_HFRx_act_trans=action_pred(SRL,h_state,act,s_l,policy)

    a_l_=torch.tensor(a_l)
    a_l_trans=torch.where(a_l_ == 0, -1, a_l_.long())
    action_score=policy.qf1(torch.cat((state,a_l_trans),-1))
    RL_HFRx_score=policy.qf1(torch.cat((state,RL_HFRx_act_trans),-1))
    
    